In [2]:
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("model_fpga_ready.h5", compile = False)

# Extract weights
weights = []
biases = []

for layer in model.layers:
    if "dense" in layer.name or "output" in layer.name:
        w, b = layer.get_weights()
        weights.append(w.astype(np.float32))
        biases.append(b.astype(np.float32))

print("Layers extracted:", len(weights))

for i in range(len(weights)):
    print(f"Layer {i+1} weights shape:", weights[i].shape)
    print(f"Layer {i+1} bias shape:", biases[i].shape)

Layers extracted: 3
Layer 1 weights shape: (54, 16)
Layer 1 bias shape: (16,)
Layer 2 weights shape: (16, 8)
Layer 2 bias shape: (8,)
Layer 3 weights shape: (8, 1)
Layer 3 bias shape: (1,)


In [3]:
def relu(x):
    return np.maximum(0, x)

In [4]:
def mlp_forward(x, weights, biases):
    # Layer 1
    z1 = np.dot(x, weights[0]) + biases[0]
    a1 = relu(z1)

    # Layer 2
    z2 = np.dot(a1, weights[1]) + biases[1]
    a2 = relu(z2)

    # Output layer
    z3 = np.dot(a2, weights[2]) + biases[2]

    return z3

In [5]:
X = np.load("data/X_windows.npy")
y = np.load("data/y_targets.npy")

# Use same split logic
split_index = int(0.8 * len(X))

X_test = X[split_index:]
y_test = y[split_index:]

In [6]:
# Take a few samples
num_samples = 5

for i in range(num_samples):
    x = X_test[i:i+1]

    tf_pred = model.predict(x, verbose=0)
    np_pred = mlp_forward(x, weights, biases)

    print(f"\nSample {i}")
    print("TensorFlow:", tf_pred[0][0])
    print("NumPy     :", np_pred[0][0])
    print("Difference:", abs(tf_pred[0][0] - np_pred[0][0]))


Sample 0
TensorFlow: 46.010353
NumPy     : 46.01036
Difference: 7.6293945e-06

Sample 1
TensorFlow: 43.743195
NumPy     : 43.743195
Difference: 0.0

Sample 2
TensorFlow: 42.07762
NumPy     : 42.07762
Difference: 0.0

Sample 3
TensorFlow: 49.533108
NumPy     : 49.533104
Difference: 3.8146973e-06

Sample 4
TensorFlow: 49.98597
NumPy     : 49.985977
Difference: 7.6293945e-06
